In [ ]:
# -*- coding: utf-8 -*-
import re, time, os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble     import RandomForestRegressor
from sklearn.metrics      import r2_score, mean_squared_error, mean_absolute_error

# ======== PARÂMETROS ========
REF_TEMP      = 10
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50

# Caminhos corrigidos (mesma pasta do script)
PKL_TREINO = "base_treino.pkl"
PKL_PROVA  = "base_prova (1).pkl"

CHUNK_SIZE = 100
RF_PARAMS = dict(
    n_estimators=450,
    max_depth=13,
    min_samples_leaf=5,
    min_samples_split=10,
    max_features="sqrt",
    bootstrap=True,
    oob_score=False,
    n_jobs=-1,
    random_state=42,
)

# Sub-bandas para métricas
N_BANDS = 4

# =====================================================
# FUNÇÕES AUXILIARES
# =====================================================
def extract_freq_hz(col_name: str):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col_name))
    return float(m.group(1)) if m else None

def get_freq_columns(df: pd.DataFrame, fmin_khz: float, fmax_khz: float):
    cols, freqs_hz = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs_hz.append(f)
    order = np.argsort(freqs_hz)
    return [cols[i] for i in order], np.array(freqs_hz, float)[order]

def metrics_block(name, y_true, y_pred):
    y1, y2 = y_true.reshape(-1), y_pred.reshape(-1)
    R2   = r2_score(y1, y2)
    RMSE = float(np.sqrt(mean_squared_error(y1, y2)))
    MAE  = float(mean_absolute_error(y1, y2))
    print(f"\n== {name} ==")
    print(f"R²={R2:.4f} | RMSE={RMSE:.6g} | MAE={MAE:.6g}")
    return {"R2":R2,"RMSE":RMSE,"MAE":MAE}

# ---- métricas extras por curva ----
def corr_per_sample(Y, Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(np.clip(num/den,-1,1))
    return np.array(out)

def sam_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        num=(y*yh).sum()
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=np.clip(num/den,-1,1)
        out.append(np.degrees(np.arccos(cosang)))
    return np.array(out)

def nrmse_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(rmse/(rng+1e-12))
    return np.array(out)

def plot_example(freqs_hz, y_orig, y_transf, y_ref, title):
    plt.figure()
    plt.plot(freqs_hz/1e3, y_orig,    label="Original")
    plt.plot(freqs_hz/1e3, y_transf,  label="Transformada (→ ref)")
    plt.plot(freqs_hz/1e3, y_ref,     label=f"Referência ({REF_TEMP} °C)")
    plt.xlabel("Frequência (kHz)"); plt.ylabel("Re{Z}")
    plt.grid(True); plt.legend(); plt.title(title); plt.show()

def plot_error_band(freqs_hz, Y_true, Y_pred, title):
    err = Y_pred - Y_true
    mu  = err.mean(axis=0); sd = err.std(axis=0)
    plt.figure()
    plt.plot(freqs_hz/1e3, mu, label="Erro médio")
    plt.fill_between(freqs_hz/1e3, mu-sd, mu+sd, alpha=0.25, label="±1 desvio")
    plt.xlabel("Frequência (kHz)"); plt.ylabel("Erro")
    plt.grid(True); plt.legend(); plt.title(title); plt.show()

# =====================================================
# CARREGAR BASES
# =====================================================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, fhz_tr = get_freq_columns(base_tr,FREQ_MIN_KHZ,FREQ_MAX_KHZ)
freq_cols_te, fhz_te = get_freq_columns(base_te,FREQ_MIN_KHZ,FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order=np.argsort(fhz); common_cols=[common_cols[i] for i in order]; fhz=fhz[order]

X_tr_full = base_tr[common_cols].to_numpy(float)
X_te      = base_te[common_cols].to_numpy(float)

ref_rows = (base_tr["temp_c"].to_numpy() == REF_TEMP)
assert ref_rows.any(), f"Não há medições a {REF_TEMP} °C no TREINO."
y_ref = np.median(base_tr.loc[ref_rows, common_cols].to_numpy(float), axis=0)

Y_tr_full = (y_ref[None,:] - X_tr_full)

# Split treino/val
X_tr_in,X_val_in,Y_tr_in,Y_val_in=train_test_split(
    X_tr_full,Y_tr_full,test_size=0.2,random_state=42,shuffle=True
)

# =====================================================
# TREINO RANDOM FOREST EM CHUNKS (com fix do width==1)
# =====================================================
def fit_predict_rf_chunks(X_tr,Y_tr,X_val,X_te,chunk_size=CHUNK_SIZE):
    n_out=Y_tr.shape[1]
    dtr_pred=np.zeros_like(Y_tr)
    dval_pred=np.zeros((X_val.shape[0],n_out))
    dte_pred =np.zeros((X_te.shape[0],n_out))
    n_chunks=int(np.ceil(n_out/chunk_size))
    print(f"[INFO] Saídas={n_out} | CHUNK_SIZE={chunk_size} | n_chunks={n_chunks}")

    for start in range(0,n_out,chunk_size):
        end=min(start+chunk_size,n_out)
        cols=slice(start,end)
        width=end-start

        rf=RandomForestRegressor(**RF_PARAMS)

        if width == 1:
            rf.fit(X_tr, Y_tr[:,cols].ravel())
            dtr=rf.predict(X_tr).reshape(-1,1)
            dvl=rf.predict(X_val).reshape(-1,1) if X_val.size else np.empty((0,1))
            dte=rf.predict(X_te ).reshape(-1,1) if X_te.size  else np.empty((0,1))
        else:
            rf.fit(X_tr, Y_tr[:,cols])
            dtr=rf.predict(X_tr)
            dvl=rf.predict(X_val) if X_val.size else np.empty((0,width))
            dte=rf.predict(X_te ) if X_te.size  else np.empty((0,width))
            if dtr.ndim==1: dtr=dtr.reshape(-1,1)
            if dvl.ndim==1: dvl=dvl.reshape(-1,1)
            if dte.ndim==1: dte=dte.reshape(-1,1)

        dtr_pred[:,cols]=dtr
        if X_val.size: dval_pred[:,cols]=dvl
        if X_te.size:  dte_pred[:,cols]=dte

        print(f"  Chunk {start}:{end} | width={width} | ok")

    return dtr_pred,dval_pred,dte_pred

t0=time.time()
dtr_pred,dval_pred,dte_pred=fit_predict_rf_chunks(X_tr_in,Y_tr_in,X_val_in,X_te,CHUNK_SIZE)
print(f"[INFO] Tempo treino+pred: {time.time()-t0:.1f}s")

# Reconstrução
Y_tr_hat=X_tr_in+dtr_pred
Y_val_hat=X_val_in+dval_pred
Y_te_hat =X_te    +dte_pred

Y_ref_tr =np.tile(y_ref,(Y_tr_hat.shape[0],1))
Y_ref_val=np.tile(y_ref,(Y_val_hat.shape[0],1))
Y_ref_te =np.tile(y_ref,(Y_te_hat.shape[0],1))

# =====================================================
# MÉTRICAS GERAIS
# =====================================================
print("\n== MÉTRICAS GERAIS ==")
m_train=metrics_block("TREINO interno",Y_ref_tr,Y_tr_hat)
m_val  =metrics_block("VALIDAÇÃO interna",Y_ref_val,Y_val_hat)
m_test =metrics_block("PROVA",Y_ref_te,Y_te_hat)

corr_vec=corr_per_sample(Y_ref_te,Y_te_hat)
sam_vec =sam_per_sample(Y_ref_te,Y_te_hat)
nrmse_vec=nrmse_per_sample(Y_ref_te,Y_te_hat)
print(f"Corr (méd.): {corr_vec.mean():.4f} | SAM° (méd.): {sam_vec.mean():.2f} | NRMSE (méd.): {nrmse_vec.mean():.4f}")

pd.DataFrame([{
    **m_test,
    "Corr_mean":float(corr_vec.mean()),
    "SAM_deg_mean":float(sam_vec.mean()),
    "NRMSE_mean":float(nrmse_vec.mean())
}]).to_csv("metrics_overall_rf.csv",index=False)

# =====================================================
# MÉTRICAS POR PEÇA
# =====================================================
if "piece_id" in base_te.columns:
    rows_by_piece={}
    for i,pid in enumerate(base_te["piece_id"]):
        rows_by_piece.setdefault(pid,[]).append(i)
    recs=[]
    for pid,rows in rows_by_piece.items():
        yt,yp=Y_ref_te[rows],Y_te_hat[rows]
        recs.append({
            "piece_id":pid,
            "R2":r2_score(yt.reshape(-1),yp.reshape(-1)),
            "RMSE":float(np.sqrt(mean_squared_error(yt,yp))),
            "MAE":float(mean_absolute_error(yt,yp)),
            "Corr":float(corr_per_sample(yt,yp).mean()),
            "SAM_deg":float(sam_per_sample(yt,yp).mean()),
            "NRMSE":float(nrmse_per_sample(yt,yp).mean()),
        })
    pd.DataFrame(recs).to_csv("metrics_by_piece_rf.csv",index=False)

# =====================================================
# MÉTRICAS POR SUB-BANDA
# =====================================================
band_edges=np.linspace(fhz.min(),fhz.max(),N_BANDS+1)
band_recs=[]
for b in range(N_BANDS):
    f0,f1=band_edges[b],band_edges[b+1]
    cols=np.where((fhz>=f0)&(fhz<=f1))[0]
    if len(cols)==0: continue
    yt,yp=Y_ref_te[:,cols],Y_te_hat[:,cols]
    band_recs.append({
        "band_idx":b,
        "fmin_hz":float(f0),"fmax_hz":float(f1),"n_cols":len(cols),
        "R2":r2_score(yt.reshape(-1),yp.reshape(-1)),
        "RMSE":float(np.sqrt(mean_squared_error(yt,yp))),
        "MAE":float(mean_absolute_error(yt,yp)),
        "Corr":float(corr_per_sample(yt,yp).mean()),
        "SAM_deg":float(sam_per_sample(yt,yp).mean()),
        "NRMSE":float(nrmse_per_sample(yt,yp).mean()),
    })
pd.DataFrame(band_recs).to_csv("metrics_by_band_rf.csv",index=False)

# =====================================================
# MÉTRICAS POR FREQUÊNCIA INDIVIDUAL
# =====================================================
freq_recs=[]
for j,f in enumerate(fhz):
    yt,yp=Y_ref_te[:,j],Y_te_hat[:,j]
    freq_recs.append({
        "freq_hz":float(f),
        "R2":r2_score(yt,yp),
        "RMSE":float(np.sqrt(mean_squared_error(yt,yp))),
        "MAE":float(mean_absolute_error(yt,yp)),
    })
pd.DataFrame(freq_recs).to_csv("metrics_by_freq_rf.csv",index=False)

# =====================================================
# PLOTS
# =====================================================
if X_te.shape[0]>0:
    plot_example(fhz,X_te[0],Y_te_hat[0],y_ref,"Exemplo PROVA idx=0")
    plot_error_band(fhz,Y_ref_te,Y_te_hat,"Erro (Transformada - Referência) — PROVA")
